In [30]:
import pandas as pd
import numpy as np

# 📡 Kapsama hesapları için
from coverage import calculate_coverage  
from path_config import PathConfig

# 📊 Performans metrikleri için
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [31]:
def run_coverage_evaluation(
    tx_lat,
    tx_lon,
    tx_height_m,
    target_column,
    azimuth_deg=0,
    h_bw_deg=65,
    v_bw_deg=10,
    n_tx=64,
    sheet_name="Series Formatted Data",
    pci_target=30,
):
    paths = PathConfig()
    dl_path = paths._processed_dir/"DL_UL_Scanner"/"merged_data_dl.parquet"
    # 2. Veriyi oku
    df = pd.read_parquet(dl_path)

    # 3. Filtreleme
    df_target = df[df["NR_UE_PCI_0"] == pci_target]
    df_target = df_target[df_target[target_column].notna()]
    # Sadece geçerli koordinatlara sahip satırları kullan
    df_target = df_target[df_target["Latitude"].notna() & df_target["Longitude"].notna()]
    # Her bir satır için coverage hesapla ve sonuçları birleştir
    results = []
    for _, row in df_target.iterrows():
        coverage_result = calculate_coverage(
            tx_lat=tx_lat,
            tx_lon=tx_lon,
            tx_height_m=tx_height_m,
            rx_lat=row["Latitude"],
            rx_lon=row["Longitude"],
            rx_height_m=0,
            azimuth_deg=azimuth_deg,
            h_bw_deg=h_bw_deg,
            v_bw_deg=v_bw_deg,
            n_tx=n_tx
        )
        try:
            pred_value = float(coverage_result[target_column].iloc[0])
        except Exception as e:
            print(f"Hata: {e}, coverage_result: {coverage_result}, target_column: {target_column}")
            pred_value = np.nan
        results.append({
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Time": row["Time"],
            "Predicted Value": pred_value,
            "Actual Value": row[target_column]
        })

    coverage_df = pd.DataFrame(results)

    # Sklearn metriklerini hesapla ve yazdır
    if not coverage_df.empty:
        y_true = coverage_df["Actual Value"]
        y_pred = coverage_df["Predicted Value"]
        print("MAE:", mean_absolute_error(y_true, y_pred))
        print("MSE:", mean_squared_error(y_true, y_pred))
        print("RMSE:", mean_squared_error(y_true, y_pred, squared=False))
        print("R2:", r2_score(y_true, y_pred))

    return coverage_df

In [32]:
# ...existing code...
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if __name__ == "__main__":
    # Her transmitter için coverage hesapla (her satır bir transmitter)
    paths = PathConfig()
    hucre_path = paths.Hucre_Bilgileri
    hucre_df = pd.read_excel(hucre_path)
    hucre_df.columns = hucre_df.columns.str.strip()  # Remove leading/trailing spaces from column names

    # Kayıt klasörü: Results/coverage
    coverage_dir = paths.results_dir / "coverage/merged/DL"
    coverage_dir.mkdir(parents=True, exist_ok=True)

    target_columns = [
        "NR_UE_RSRP_0",
        "NR_UE_Timing_Advance",
        "NR_UE_Pathloss_DL_0"
    ]

    metrics_list = []

    for idx, row in hucre_df.iterrows():
        tx_lat = row["Latitude"]
        tx_lon = row["Longitude"]
        tx_height_m = row["Height [m]"]
        azimuth_deg = row["Azimuth [°]"]
        h_bw_deg = row["Horizontal Beamwidth [°]"]
        v_bw_deg = row["Vertical Beamwidth [°]"]
        pci_target = int(row["PCI"])
        mimo_str = str(row["MIMO"])
        n_tx = int(mimo_str[:2]) if mimo_str[:2].isdigit() else 64  # 64/32 gibi başlar

        print(f"\n--- Hücre {idx+1} (PCI: {pci_target}) için transmitter parametreleri ---")
        print(f"tx_lat: {tx_lat}, tx_lon: {tx_lon}, tx_height_m: {tx_height_m}, azimuth_deg: {azimuth_deg}, h_bw_deg: {h_bw_deg}, v_bw_deg: {v_bw_deg}, n_tx: {n_tx}")

        for target_col in target_columns:
            print(f"\n--- {target_col} için sonuçlar ---")
            df_result = run_coverage_evaluation(
                tx_lat=tx_lat,
                tx_lon=tx_lon,
                tx_height_m=tx_height_m,
                azimuth_deg=azimuth_deg,
                h_bw_deg=h_bw_deg,
                v_bw_deg=v_bw_deg,
                n_tx=n_tx,
                pci_target=pci_target,
                sheet_name="Series Formatted Data",
                target_column=target_col
            )
            print(df_result.head())

            # Sonuçları Results/coverage klasörüne kaydet
            csv_filename = f"coverage_{pci_target}_{target_col}.csv"
            csv_path = coverage_dir / csv_filename
            df_result.to_csv(csv_path, index=False)

            # Metrikleri hesapla ve listeye ekle
            if not df_result.empty:
                y_true = df_result["Actual Value"]
                y_pred = df_result["Predicted Value"]
                metrics_list.append({
                    "PCI": pci_target,
                    "Target Column": target_col,
                    "MAE": mean_absolute_error(y_true, y_pred),
                    "MSE": mean_squared_error(y_true, y_pred),
                    "RMSE": mean_squared_error(y_true, y_pred, squared=False),
                    "R2": r2_score(y_true, y_pred)
                })

    # Tüm metrikleri tek bir csv'ye kaydet
    if metrics_list:
        metrics_df = pd.DataFrame(metrics_list)
        metrics_df.to_csv(coverage_dir / "coverage_metrics.csv", index=False)
# ...existing code...<


--- Hücre 1 (PCI: 30) için transmitter parametreleri ---
tx_lat: 41.1073413, tx_lon: 29.0233668, tx_height_m: 10, azimuth_deg: 50, h_bw_deg: 65, v_bw_deg: 10.0, n_tx: 64

--- NR_UE_RSRP_0 için sonuçlar ---


TypeError: unsupported operand type(s) for -: 'str' and 'float'

In [ ]:
paths = PathConfig()
dl_path = paths.dl_path
df = pd.read_excel(dl_path, sheet_name="Series Formatted Data")
df_target = df[df["NR_UE_Timing_Advance"].notna()]
df_target = df_target[df_target["NR_UE_PCI_0"] == 30]

print("\nHedef PCI 30 olan ve Pathloss DL değeri olan ilk 5 satır:")
print(df_target.head())




Hedef PCI 30 olan ve Pathloss DL değeri olan ilk 5 satır:
Empty DataFrame
Columns: [Message, Time, Longitude, Latitude, Technology_Mode, NR_UE_PCI_0, NR_UE_RSRP_0, NR_UE_RSRQ_0, NR_UE_SINR_0, NR_UE_Nbr_PCI_0, NR_UE_Nbr_PCI_1, NR_UE_Nbr_PCI_2, NR_UE_Nbr_PCI_3, NR_UE_Nbr_PCI_4, NR_UE_Nbr_RSRP_0, NR_UE_Nbr_RSRP_1, NR_UE_Nbr_RSRP_2, NR_UE_Nbr_RSRP_3, NR_UE_Nbr_RSRP_4, NR_UE_Nbr_RSRQ_0, NR_UE_Nbr_RSRQ_1, NR_UE_Nbr_RSRQ_2, NR_UE_Nbr_RSRQ_3, NR_UE_Nbr_RSRQ_4, NR_UE_Timing_Advance, NR_UE_Pathloss_DL_0, NR_UE_Throughput_PDCP_DL, App_Throughput_DL, NR_UE_NACK_Rate_DL_0, NR_UE_Ack_As_Nack_DL_0, NR_UE_MCS_DL_0, NR_UE_RB_Num_DL_0, NR_UE_Modulation_Avg_DL_0, NR_UE_RI_DL_0, NR_UE_BLER_DL_0, NR_UE_CCE_AggregationLev_0, NR_UE_Power_Tx_PUSCH_0, NR_UE_Power_Tx_PRACH_0, NR_UE_NACK_Rate_UL_0, NR_UE_RACH_Attempt, NR_UE_RACH_OK, NR_UE_RACH_Fail, NR_UE_RACH_Procedure_Count, NR_UE_RRCReEstAttempt, NR_UE_RRCReEstFail, NR_UE_RRCReEst_EndResult, NR_UE_RRCConnectionAttempt, NR_UE_RRCConnectionSetupOk, Unnamed: 48